# Qwen3.5-0.8B × TinyCeNN Memory Fusion — Sequential Acceptance

Runs `Qwen/Qwen3.5-0.8B`. Native Qwen3.5 linear-attention/Gated-DeltaNet layers stay unchanged; TinyCeNN Memory Fusion is trained only on full-attention anchors **3, 7, 11, 15, 19, 23**.

This version adds a saved-state **progress dashboard** and real **user-chat comparisons** against the original model. Accepted-only adapter weights and resumable training state are still published to Hugging Face.


In [1]:
import os,sys,subprocess,shutil,json
from pathlib import Path
assert subprocess.run(["nvidia-smi"],check=False).returncode==0,"Enable GPU runtime"
REPO=Path("/content/TinyCeNN-LM")
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(["git","clone","--depth","1","https://github.com/vtavakkoli/TinyCeNN-LM.git",str(REPO)],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","transformers==5.17.0","datasets","huggingface_hub","accelerate","safetensors","pytest","pandas"],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-e",str(REPO),"--no-deps"],check=True)
for p in (str(REPO),str(REPO/"src")):
    if p not in sys.path: sys.path.insert(0,p)
os.environ["PYTHONPATH"]=os.pathsep.join([str(REPO),str(REPO/"src")])
print("✅ repo",subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip())


✅ repo 6322303dd6cfc0485955d8d0af8f1988ee8d2135


In [2]:
from google.colab import drive
from huggingface_hub import HfApi
from transformers import AutoConfig
drive.mount("/content/drive")
BASE_MODEL="Qwen/Qwen3.5-0.8B"
MODEL_REVISION=HfApi().model_info(BASE_MODEL).sha
FEATURE_DIM=32; MEMORY_RANK=64; CONTEXT=128; PROBE_CONTEXT=128; SEED=73
MAX_ROUNDS_PER_RUN=4; MAX_LAYER_STEPS=300; CHECK_EVERY=25; RESET_PROGRESS=False
HF_MODEL_REPO="vtava/Qwen3.5-0.8B-MemoryFusion"; HF_PRIVATE=False; PUBLISH_TO_HF=True
OUTPUT_DIR=Path("/content/drive/MyDrive/TinyCeNN-LM/qwen35-0.8b-memory-fusion-sequential-r64")
if RESET_PROGRESS and OUTPUT_DIR.exists(): shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
cfg=AutoConfig.from_pretrained(BASE_MODEL,revision=MODEL_REVISION).get_text_config(decoder=True)
TARGET_LAYERS=[i for i,k in enumerate(cfg.layer_types) if k=="full_attention"]
LINEAR_LAYERS=[i for i,k in enumerate(cfg.layer_types) if k=="linear_attention"]
assert TARGET_LAYERS==[3,7,11,15,19,23],TARGET_LAYERS
print({"revision":MODEL_REVISION,"targets":TARGET_LAYERS,"native_linear":LINEAR_LAYERS,"output":str(OUTPUT_DIR),"hf_repo":HF_MODEL_REPO})


Mounted at /content/drive


config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

{'revision': '2fc06364715b967f1860aea9cf38778875588b17', 'targets': [3, 7, 11, 15, 19, 23], 'native_linear': [0, 1, 2, 4, 5, 6, 8, 9, 10, 12, 13, 14, 16, 17, 18, 20, 21, 22], 'output': '/content/drive/MyDrive/TinyCeNN-LM/qwen35-0.8b-memory-fusion-sequential-r64', 'hf_repo': 'vtava/Qwen3.5-0.8B-MemoryFusion'}


In [3]:
from huggingface_hub import HfApi,login,notebook_login
from google.colab import userdata
try: token=userdata.get("HF_TOKEN")
except Exception: token=None
if token:
    os.environ["HF_TOKEN"]=token; login(token=token,add_to_git_credential=False)
else: notebook_login()
print("✅ HF user",HfApi().whoami().get("name"))


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ HF user vtava


In [4]:
env=dict(os.environ,CUDA_VISIBLE_DEVICES="",OMP_NUM_THREADS="1",MKL_NUM_THREADS="1")
env["TINYCENN_PARENT_BACKUP_ACTIVE"]="1"
r=subprocess.run([sys.executable,"-m","pytest","-q","tests/test_qwen3_5_memory_fusion.py"],cwd=REPO,env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(r.stdout)
if r.returncode: raise RuntimeError(f"preflight failed: {r.returncode}")
print("✅ preflight passed")


....                                                                     [100%]
4 passed in 20.33s

✅ preflight passed


In [5]:
import pandas as pd
from IPython.display import display,HTML
TH={"nmse":.20,"cosine":.90,"incremental_delta_nll":.015,"cumulative_delta_nll":.05}
def _j(name):
    p=OUTPUT_DIR/name
    return json.loads(p.read_text()) if p.exists() else {}
def show_progress():
    s,p,ip,f=(_j("sequential_run_status.json"),_j("sequential_progress.json"),_j("sequential_in_progress.json"),_j("sequential_training_report.json"))
    accepted=[int(x) for x in p.get("accepted_layers",s.get("accepted_layers",[]))]
    target=[int(x) for x in p.get("target_layers",s.get("target_layers",TARGET_LAYERS))]
    current=ip.get("current_layer",s.get("current_layer")); rounds=int(ip.get("rounds_completed",s.get("rounds_completed",0)) or 0)
    pct=100*len(accepted)/max(1,len(target))
    pills=[]
    for L in target:
        bg,label=("#d1fae5",f"✓ L{L}") if L in accepted else (("#fef3c7",f"▶ L{L}") if current is not None and int(current)==L else ("#e5e7eb",f"L{L}"))
        pills.append(f"<span style='padding:6px 10px;margin:3px;border-radius:8px;background:{bg};display:inline-block'>{label}</span>")
    display(HTML(f"<div style='border:1px solid #bbb;border-radius:12px;padding:14px'><h3>Qwen3.5 Memory Fusion progress</h3>{''.join(pills)}<p><b>{len(accepted)}/{len(target)}</b> anchors accepted ({pct:.1f}%)</p><div style='height:15px;background:#ddd;border-radius:8px'><div style='height:15px;width:{pct:.1f}%;background:#22c55e;border-radius:8px'></div></div><p>Current anchor: <b>{current}</b> | rounds completed: <b>{rounds}</b> | status: <b>{s.get('status',f.get('status','not started'))}</b></p></div>"))
    reports=[]
    for x in (p,ip,f):
        if x.get("layer_reports"): reports=x["layer_reports"]
    if reports:
        df=pd.DataFrame(reports); cols=[c for c in ["layer","round","accepted","steps","nmse","cosine","probe_nll","incremental_delta_nll","cumulative_delta_nll"] if c in df]
        display(df[cols].tail(20))
        last=reports[-1]
        display(pd.DataFrame([
          ["NMSE",last.get("nmse"),"≤ .20",last.get("nmse",9)<=TH["nmse"]],
          ["Cosine",last.get("cosine"),"≥ .90",last.get("cosine",-9)>=TH["cosine"]],
          ["Incremental ΔNLL",last.get("incremental_delta_nll"),"≤ +.015",last.get("incremental_delta_nll",9)<=TH["incremental_delta_nll"]],
          ["Cumulative ΔNLL",last.get("cumulative_delta_nll"),"≤ +.050",last.get("cumulative_delta_nll",9)<=TH["cumulative_delta_nll"]],
        ],columns=["metric","value","required","pass"]))
    return {"accepted":accepted,"current":current,"rounds":rounds}
PROGRESS_BEFORE=show_progress()


In [6]:
cmd=[sys.executable,"-u",str(REPO/"scripts"/"train_qwen35_memory_fusion_sequential.py"),
"--base-model",BASE_MODEL,"--model-revision",MODEL_REVISION,"--output-dir",str(OUTPUT_DIR),
"--feature-dim",str(FEATURE_DIM),"--memory-rank",str(MEMORY_RANK),"--context-length",str(CONTEXT),"--probe-context",str(PROBE_CONTEXT),
"--seed",str(SEED),"--min-layer-steps","50","--max-layer-steps",str(MAX_LAYER_STEPS),"--check-every",str(CHECK_EVERY),
"--layer-lr","0.0002","--teacher-alpha-start","0.9","--teacher-alpha-end","0.0",
"--accept-nmse","0.2","--accept-cosine","0.9","--accept-incremental-delta-nll","0.015","--accept-cumulative-delta-nll","0.05",
"--max-runtime-minutes","240","--resume","--strict-acceptance"]
print("▶ training/resume; targets:",TARGET_LAYERS); print(" ".join(cmd),flush=True)
run_env=dict(os.environ); run_env["SEQUENTIAL_MAX_ROUNDS_PER_RUN"]=str(MAX_ROUNDS_PER_RUN)
r=subprocess.run(cmd,cwd=REPO,env=run_env)
if r.returncode: raise RuntimeError(f"trainer failed: {r.returncode}")
print("✅ trainer returned normally")
print("Updated progress:")
PROGRESS_AFTER=show_progress()


▶ training/resume; targets: [3, 7, 11, 15, 19, 23]
/usr/bin/python3 -u /content/TinyCeNN-LM/scripts/train_qwen35_memory_fusion_sequential.py --base-model Qwen/Qwen3.5-0.8B --model-revision 2fc06364715b967f1860aea9cf38778875588b17 --output-dir /content/drive/MyDrive/TinyCeNN-LM/qwen35-0.8b-memory-fusion-sequential-r64 --feature-dim 32 --memory-rank 64 --context-length 128 --probe-context 128 --seed 73 --min-layer-steps 50 --max-layer-steps 300 --check-every 25 --layer-lr 0.0002 --teacher-alpha-start 0.9 --teacher-alpha-end 0.0 --accept-nmse 0.2 --accept-cosine 0.9 --accept-incremental-delta-nll 0.015 --accept-cumulative-delta-nll 0.05 --max-runtime-minutes 240 --resume --strict-acceptance


KeyboardInterrupt: 

In [7]:
import torch,gc
from transformers import AutoTokenizer,Qwen3_5ForCausalLM
from tinycenn_lm.qwen3_5_memory_fusion import Qwen35MemoryFusionConfig,replace_attention_layers,load_selected_attention_state,structural_summary
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE=torch.bfloat16 if DEVICE.type=="cuda" and torch.cuda.is_bf16_supported() else (torch.float16 if DEVICE.type=="cuda" else torch.float32)
pp=OUTPUT_DIR/"sequential_progress.pt"; payload=torch.load(pp,map_location="cpu",weights_only=False) if pp.exists() else None
accepted=[int(x) for x in payload.get("accepted_layers",[])] if payload else []
mf_cfg=Qwen35MemoryFusionConfig.from_dict(payload["config"]) if payload else Qwen35MemoryFusionConfig(feature_dim=FEATURE_DIM,memory_rank=MEMORY_RANK)
tok=AutoTokenizer.from_pretrained(BASE_MODEL,revision=MODEL_REVISION)
original=Qwen3_5ForCausalLM.from_pretrained(BASE_MODEL,revision=MODEL_REVISION,dtype=DTYPE,attn_implementation="sdpa",low_cpu_mem_usage=True).to(DEVICE).eval()
adapted=Qwen3_5ForCausalLM.from_pretrained(BASE_MODEL,revision=MODEL_REVISION,dtype=DTYPE,attn_implementation="sdpa",low_cpu_mem_usage=True).to(DEVICE).eval()
if accepted:
    replace_attention_layers(adapted,mf_cfg,accepted); load_selected_attention_state(adapted,payload["attention_state"],accepted)
original.config.use_cache=False; adapted.config.use_cache=False
print("Accepted:",accepted); print(json.dumps(structural_summary(adapted),indent=2))
USER_CHATS=[
{"name":"factual","messages":[{"role":"system","content":"You are helpful and concise."},{"role":"user","content":"What is the capital of Austria? Answer in one sentence."}]},
{"name":"explanation","messages":[{"role":"system","content":"Explain clearly for non-experts."},{"role":"user","content":"Why does the sky look blue? Explain it to a 10-year-old in two sentences."}]},
{"name":"coding","messages":[{"role":"system","content":"You are a careful Python assistant."},{"role":"user","content":"Write a small Python function is_prime(n) and explain the key idea briefly."}]},
{"name":"reasoning","messages":[{"role":"system","content":"Show the calculation briefly, then answer."},{"role":"user","content":"A train travels 120 km in 1.5 hours. What is its average speed in km/h?"}]},
{"name":"multi_turn","messages":[{"role":"system","content":"You are a practical software architecture assistant."},{"role":"user","content":"I am building a small Python REST API."},{"role":"assistant","content":"What would you like to improve?"},{"role":"user","content":"Give me three concrete ways to make it more reliable in production."}]}
]
@torch.no_grad()
def chat(model,msgs):
    text=tok.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True); x=tok(text,return_tensors="pt").to(DEVICE)
    y=model.generate(**x,max_new_tokens=96,do_sample=False,use_cache=False,pad_token_id=tok.eos_token_id)
    return tok.decode(y[0,x["input_ids"].shape[1]:],skip_special_tokens=True).strip()
examples=[]
for i,c in enumerate(USER_CHATS,1):
    a,b=chat(original,c["messages"]),chat(adapted,c["messages"])
    examples.append({"name":c["name"],"messages":c["messages"],"original_reply":a,"memory_fusion_reply":b,"accepted_layers":accepted})
    print("\n"+"="*100+f"\nCHAT {i}: {c['name']}")
    for m in c["messages"]: print(f"{m['role'].upper()}: {m['content']}")
    print("\nORIGINAL QWEN3.5:\n",a); print("\nMEMORY FUSION:\n",b)
(OUTPUT_DIR/"chat_examples.json").write_text(json.dumps(examples,indent=2,ensure_ascii=False),encoding="utf-8")
print("✅ chat comparison saved")


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Accepted: []
{
  "memory_fusion_layers": [],
  "remaining_full_attention_layers": [
    3,
    7,
    11,
    15,
    19,
    23
  ],
  "native_linear_attention_layers": [
    0,
    1,
    2,
    4,
    5,
    6,
    8,
    9,
    10,
    12,
    13,
    14,
    16,
    17,
    18,
    20,
    21,
    22
  ]
}


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.



CHAT 1: factual
SYSTEM: You are helpful and concise.
USER: What is the capital of Austria? Answer in one sentence.

ORIGINAL QWEN3.5:
 Vienna is the capital of Austria.

MEMORY FUSION:
 Vienna is the capital of Austria.

CHAT 2: explanation
SYSTEM: Explain clearly for non-experts.
USER: Why does the sky look blue? Explain it to a 10-year-old in two sentences.

ORIGINAL QWEN3.5:
 The sky looks blue because the sun shines on the Earth, and the blue sky is actually a thin layer of tiny blue dust floating in the air. When sunlight hits these tiny particles, they bounce back in all directions, but the blue ones bounce back the most, making the sky appear blue.

MEMORY FUSION:
 The sky looks blue because the sun shines on the Earth, and the blue sky is actually a thin layer of tiny blue dust floating in the air. When sunlight hits these tiny particles, they bounce back in all directions, but the blue ones bounce back the most, making the sky appear blue.

CHAT 3: coding
SYSTEM: You are a ca

In [8]:
from tinycenn_lm.qwen3_5_memory_fusion import save_adapter
PACKAGE=Path("/content/qwen35-memory-fusion-hf")
if PACKAGE.exists(): shutil.rmtree(PACKAGE)
(PACKAGE/"training").mkdir(parents=True)
pp=OUTPUT_DIR/"sequential_progress.pt"; payload=torch.load(pp,map_location="cpu",weights_only=False) if pp.exists() else None
accepted=[int(x) for x in payload.get("accepted_layers",[])] if payload else []
mf_cfg=Qwen35MemoryFusionConfig.from_dict(payload["config"]) if payload else Qwen35MemoryFusionConfig(feature_dim=FEATURE_DIM,memory_rank=MEMORY_RANK)
if accepted: save_adapter(adapted,PACKAGE,config=mf_cfg,base_model=BASE_MODEL,accepted_layers=accepted,metadata={"base_revision":MODEL_REVISION,"source_repo":"vtavakkoli/TinyCeNN-LM"})
for n in ["sequential_run_status.json","sequential_progress.json","sequential_in_progress.json","sequential_training_report.json","chat_examples.json"]:
    s=OUTPUT_DIR/n
    if s.exists(): shutil.copy2(s,PACKAGE/n)
for n in ["sequential_in_progress.pt","sequential_progress.pt","qwen35_memory_fusion_full_state.pt"]:
    s=OUTPUT_DIR/n
    if s.exists(): shutil.copy2(s,PACKAGE/"training"/n)
reports=[]
for n in ["sequential_progress.json","sequential_in_progress.json"]:
    s=OUTPUT_DIR/n
    if s.exists(): reports=json.loads(s.read_text()).get("layer_reports",reports)
acc=[r for r in reports if r.get("accepted")]
rows=["| Layer | NMSE | Cosine | ΔNLL inc | ΔNLL total |","|---:|---:|---:|---:|---:|"]+[f"| {r['layer']} | {r['nmse']:.5f} | {r['cosine']:.5f} | {r['incremental_delta_nll']:+.5f} | {r['cumulative_delta_nll']:+.5f} |" for r in acc]
table="\n".join(rows) if acc else "_No accepted anchor yet._"
card=f'''---
library_name: transformers
base_model: {BASE_MODEL}
license: apache-2.0
tags: [qwen3.5, tinycenn, memory-fusion, recurrent-attention]
---
# Qwen3.5-0.8B × TinyCeNN Memory Fusion

Accepted anchors: `{accepted}`
Target anchors: `{TARGET_LAYERS}`
Base revision: `{MODEL_REVISION}`

## Acceptance results
{table}

## Progress and chat comparison
- `sequential_progress.json`: saved sequential progress.
- `chat_examples.json`: deterministic original-vs-Memory-Fusion user-chat comparisons.
- `training/sequential_in_progress.pt`: resumable current layer; may be unaccepted.

Acceptance gates: NMSE ≤ 0.20, cosine ≥ 0.90, incremental ΔNLL ≤ 0.015, cumulative ΔNLL ≤ 0.05.
'''
(PACKAGE/"README.md").write_text(card,encoding="utf-8")
api=HfApi()
if PUBLISH_TO_HF:
    api.create_repo(HF_MODEL_REPO,repo_type="model",private=HF_PRIVATE,exist_ok=True)
    api.upload_folder(repo_id=HF_MODEL_REPO,repo_type="model",folder_path=str(PACKAGE),commit_message=f"Update Qwen3.5 Memory Fusion accepted={accepted}")
    print("✅ published",f"https://huggingface.co/{HF_MODEL_REPO}")
else: print("Package ready:",PACKAGE)


✅ published https://huggingface.co/vtava/Qwen3.5-0.8B-MemoryFusion
